In [ ]:
# Paper 2: wav2vec 2.0: A Framework for Self-Supervised Learning of Speech Representations
import torch
import librosa
import os
import re
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def buckwalter_to_arabic(text):
    buckwalter_map = {
        '\'': 'ء', '>': 'أ', '&': 'ؤ', '<': 'إ', '}': 'ئ', 'A': 'ا', 'b': 'ب', 'p': 'ة', 't': 'ت', 'v': 'ث',
        'j': 'ج', 'H': 'ح', 'x': 'خ', 'd': 'د', '*': 'ذ', 'r': 'ر', 'z': 'ز', 's': 'س', '$': 'ش', 'S': 'ص',
        'D': 'ض', 'T': 'ط', 'Z': 'ظ', 'E': 'ع', 'g': 'غ', '_': 'ـ', 'f': 'ف', 'q': 'ق', 'k': 'ك', 'l': 'ل',
        'm': 'م', 'n': 'ن', 'h': 'ه', 'w': 'و', 'Y': 'ى', 'y': 'ي', 'F': 'ً', 'N': 'ٌ', 'K': 'ٍ', 'a': 'َ',
        'u': 'ُ', 'i': 'ِ', '~': 'ّ', 'o': 'ْ', '`': 'ٰ', '{': 'ٱ', '/': ' '
    }

    text = text.replace('/', ' ')

    for bw_char, ar_char in buckwalter_map.items():
        text = text.replace(bw_char, ar_char)
    return text

def setup_model(model_name="elgeish/wav2vec2-large-xlsr-53-arabic"):
    print(f"Loading model: {model_name}...")
    try:
        processor = Wav2Vec2Processor.from_pretrained(model_name)
        model = Wav2Vec2ForCTC.from_pretrained(model_name)
        return processor, model
    except Exception as e:
        print(f"Failed to load model {model_name}. Error: {e}")
        return None, None

def transcribe_audio(audio_path, processor, model):
    
    if not os.path.exists(audio_path):
        return f"Error: File {audio_path} not found."

    speech, samplerate = librosa.load(audio_path, sr=16000)

    inputs = processor(speech, return_tensors="pt", sampling_rate=16000)
    input_values = inputs.input_values

    with torch.no_grad():
        logits = model(input_values).logits

    predicted_ids = torch.argmax(logits, dim=-1)

    raw_transcription = processor.batch_decode(predicted_ids)[0]

    arabic_transcription = buckwalter_to_arabic(raw_transcription)

    return arabic_transcription

if __name__ == "__main__":
    MODEL_ID = "elgeish/wav2vec2-large-xlsr-53-arabic"

    proc, mod = setup_model(MODEL_ID)

    if proc and mod:
        print("\n[Success] Model loaded successfully.")
        print("-" * 30)

        audio_path = ""

        if IN_COLAB:
            print("Please upload your Arabic audio file (.wav, .mp3, etc.):")
            uploaded = files.upload()
            if uploaded:
                audio_path = list(uploaded.keys())[0]
        else:
            audio_path = input("Please enter the path to your audio file: ").strip()

        if audio_path and os.path.exists(audio_path):
            print(f"\nProcessing: {audio_path}...")
            result = transcribe_audio(audio_path, proc, mod)
            print("\n" + "="*20)
            print("Final Transcription (Arabic):")
            print(result)
            print("="*20)
        else:
            print("\nNo file uploaded or path is incorrect.")

    else:
        print("\n[Error] Failed to initialize model.")


Loading model: elgeish/wav2vec2-large-xlsr-53-arabic...


Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]


[Success] Model loaded successfully.
------------------------------
Please upload your Arabic audio file (.wav, .mp3, etc.):


Saving ar_00000.wav to ar_00000 (1).wav

Processing: ar_00000 (1).wav...

Final Transcription (Arabic):
لِأَنَّهُ لَا يَرَى أَنَّهُ عَلَى السَّفَهِ ثُمَّن بَعْدِ ذَلِكَ حَدِيثٌ مُنْتَشِرٌَ
